In [1]:
import pandas as pd
import numpy as np
import surprise
from surprise.model_selection import train_test_split

In [6]:
ratings = pd.read_csv("C:/Users/PGCP-AI.STUDENTSDC/Downloads/Cases_Rec_Sys/ml-100k/u.data",sep="\t" ,names =['uid','iid','rating','ts'])
ratings.drop("ts",axis=1,inplace=True)
ratings.head()

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [7]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings["rating"].max()
lowest_rating, highest_rating

(np.int64(1), np.int64(5))

In [8]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

similarity_options  = {"name":"cosine","user_based":True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())



Computing the cosine similarity matrix...
Done computing similarity matrix.


In [9]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
from surprise.model_selection import train_test_split




In [10]:
param_grid = {"n_epochs":[50],
              'n_factors':[100,50],
             'reg_all':np.linspace(0.01,0.8,5)}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD,param_grid,measures = ["rmse","mae"],cv=kfold, joblib_verbose=3,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:   14.4s remaining:    5.6s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   16.3s finished


In [11]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])


0.9434147147651697
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.20750000000000002)}


In [12]:
# Building bets model on whole data
algo =gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

In [13]:
iids = ratings["iid"].unique()
iids

array([ 242,  302,  377, ..., 1637, 1630, 1641], shape=(1682,))

In [14]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [15]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]

No of items rated by user 50 : 24 

Items not rated by the user or those items for which the expected ratings are to be predicted :  [   1    2    3 ... 1680 1681 1682]


In [16]:
testset = [[user,iid,0.] for iid in iids_ro_pred]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1424,1449,4.451800
391,408,4.344662
162,169,4.302831
305,318,4.300105
61,64,4.274260
465,483,4.250987
110,114,4.241338
10,12,4.222548
582,603,4.197681
171,178,4.185011
